# Complementos de Investigação Operacional — capítulo 2: Planeamento de projetos (CPM, PERT, Gantt e crashing)
### Exemplos em Python

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aguilarmadeira/applied-operations-research-course/blob/main/code/python/notebooks/02_project_planning.ipynb)

Cada secção corre o exemplo `ex….py` do método e compara os resultados com os slides (no fim: *confere com os slides: sim*). Comece pela célula **Preparação**; no Colab, *Runtime → Run all* corre tudo.

Os slides usam vírgula decimal; aqui usa-se o ponto. Este caderno é gerado a partir dos ficheiros `ex….py` por `make_notebooks.py`.

*Examples for Chapter 2 of the course "Complements of Operational Research" (ISEL, J. F. A. Madeira). Code comments and printed messages are in Portuguese; the English slides are in [`notes/en/`](https://github.com/aguilarmadeira/applied-operations-research-course/tree/main/notes/en).*

## Preparação

In [1]:
# Preparação: põe as funções da UC no caminho do Python.
# No Colab (ou noutra pasta), clona o repositório; dentro do repositório, usa code/python.
import os, sys
LOCAL = [p for p in ("..", os.path.join("..", "..")) if os.path.isfile(os.path.join(p, "uc_setup.py"))]
if LOCAL:
    RAIZ = os.path.abspath(LOCAL[0])
else:
    if not os.path.isdir("applied-operations-research-course"):
        !git clone -q --depth 1 https://github.com/aguilarmadeira/applied-operations-research-course
    RAIZ = os.path.abspath(os.path.join("applied-operations-research-course", "code", "python"))
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
import uc_setup  # acrescenta as pastas dos métodos e common/
print("Código da UC pronto.")

Código da UC pronto.


## 2.1 Planeamento de projetos: CPM

Exemplo [`ex02_1_cpm.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/02_project_planning/02_1_cpm/ex02_1_cpm.py) · funções da UC: [`pm.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/02_project_planning/02_1_cpm/pm.py)

```text
Reproduz os exemplos do deck 2.1 (planeamento de projetos: CPM).

Exemplo do capítulo: renovar um laboratório (8 atividades, soma das durações 21 semanas).
Slide «Passagem para a frente e para trás»: T = 13 semanas, caminho crítico A, C, D, F, H;
folga 1 em B, E e G.  Slide «Caminho crítico = caminho mais longo»: 12, 13, 12, 12.
Slide «O que fazer com as folgas?»: se B atrasar 2 semanas, o projeto passa a 14.
Para resolver na aula: Exemplo 1 (T = 15, crítico A, C, E, G, H) e
Exemplo 2, construção de uma obra (Hillier & Lieberman): T = 44, sem multa nem prémio.
```

In [2]:
from pm import topo, cpm, caminhos


def tabela(titulo, acts, dur=None):
    """Imprime as quatro datas e a folga de cada atividade, o caminho crítico e os caminhos."""
    ES, EF, LS, LF, F, T = cpm(acts, dur)
    d = dur or {a: v[1] for a, v in acts.items()}
    crit = [a for a in acts if abs(F[a]) < 1e-9]
    print("\n" + titulo)
    print("  ativ.  dur   ES   EF   LS   LF  folga")
    for a in topo(acts):
        print("  %-5s %4g %4g %4g %4g %4g %6g" % (a, d[a], ES[a], EF[a], LS[a], LF[a], F[a]))
    print("  duração do projeto: %g semanas; caminho crítico: %s" % (T, ", ".join(crit)))
    cam = [("--".join(p), sum(d[x] for x in p)) for p in caminhos(acts)]
    print("  caminhos: " + "; ".join("%s %g" % c for c in cam))
    return F, T, crit, cam


ok = True
print("Complementos de IO — deck 2.1: planeamento de projetos (CPM)")

# ------------------------------------------------------------ o laboratório
LAB = {"A": ([], 3), "B": (["A"], 4), "C": (["A"], 2), "D": (["C"], 3),
       "E": (["C"], 2), "F": (["B", "D", "E"], 3), "G": (["D"], 2), "H": (["F", "G"], 2)}
soma = sum(v[1] for v in LAB.values())
F, T, crit, cam = tabela("Laboratório (soma das durações: %g semanas)" % soma, LAB)
ES, EF, LS, LF, _, _ = cpm(LAB)
ok &= (soma == 21 and T == 13 and crit == ["A", "C", "D", "F", "H"]
       and [F[a] for a in LAB] == [0, 1, 0, 0, 1, 0, 1, 0]
       and [ES[a] for a in LAB] == [0, 3, 3, 5, 5, 8, 8, 11]
       and [LF[a] for a in LAB] == [3, 8, 5, 8, 8, 11, 11, 13]
       and cam == [("A--B--F--H", 12), ("A--C--D--F--H", 13), ("A--C--D--G--H", 12), ("A--C--E--F--H", 12)])

# B atrasa 2 semanas
d = {a: v[1] for a, v in LAB.items()}
d["B"] = 6
ES6, EF6, LS6, LF6, F6, T6 = cpm(LAB, d)
crit6 = [a for a in LAB if abs(F6[a]) < 1e-9]
print("\nB atrasa 2 semanas (B = 6): EF_B = %g > LS_F = %g; T = %g semanas; crítico: %s"
      % (EF6["B"], LS["F"], T6, ", ".join(crit6)))
ok &= EF6["B"] == 9 and LS["F"] == 8 and T6 == 14 and crit6 == ["A", "B", "F", "H"]

# ------------------------------------------------------------ Exemplo 1
EX1 = {"A": ([], 2), "B": ([], 3), "C": (["A"], 2), "D": (["A", "B"], 4),
       "E": (["C"], 4), "F": (["C"], 3), "G": (["D", "E"], 5), "H": (["F", "G"], 2)}
F1, T1, crit1, cam1 = tabela("Exemplo 1", EX1)
ok &= (T1 == 15 and crit1 == ["A", "C", "E", "G", "H"]
       and [F1[a] for a in EX1] == [0, 1, 0, 1, 0, 6, 0, 0]
       and cam1 == [("A--C--E--G--H", 15), ("A--C--F--H", 9), ("A--D--G--H", 13), ("B--D--G--H", 14)])

# ------------------------------------------------------------ Exemplo 2 (Hillier & Lieberman)
HL = {"A": ([], 2), "B": (["A"], 4), "C": (["B"], 10), "D": (["C"], 6), "E": (["C"], 4),
      "F": (["E"], 5), "G": (["D"], 7), "H": (["E", "G"], 9), "I": (["C"], 7), "J": (["F", "I"], 8),
      "K": (["J"], 4), "L": (["J"], 5), "M": (["H"], 2), "N": (["K", "L"], 6)}
F2, T2, crit2, cam2 = tabela("Exemplo 2 (construção de uma obra, Hillier & Lieberman)", HL)
simnao = {False: "não", True: "sim"}
print("  multa (T > 47)? %s;  prémio (T < 40)? %s;  para o prémio é preciso retirar %g semanas"
      % (simnao[T2 > 47], simnao[T2 < 40], T2 - 39))
ok &= (T2 == 44 and crit2 == ["A", "B", "C", "E", "F", "J", "L", "N"]
       and [F2[a] for a in "DGHIKM"] == [4, 4, 4, 2, 1, 4] and not T2 > 47 and not T2 < 40)

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Complementos de IO — deck 2.1: planeamento de projetos (CPM)

Laboratório (soma das durações: 21 semanas)
  ativ.  dur   ES   EF   LS   LF  folga
  A        3    0    3    0    3      0
  B        4    3    7    4    8      1
  C        2    3    5    3    5      0
  D        3    5    8    5    8      0
  E        2    5    7    6    8      1
  F        3    8   11    8   11      0
  G        2    8   10    9   11      1
  H        2   11   13   11   13      0
  duração do projeto: 13 semanas; caminho crítico: A, C, D, F, H
  caminhos: A--B--F--H 12; A--C--D--F--H 13; A--C--D--G--H 12; A--C--E--F--H 12

B atrasa 2 semanas (B = 6): EF_B = 9 > LS_F = 8; T = 14 semanas; crítico: A, B, F, H

Exemplo 1
  ativ.  dur   ES   EF   LS   LF  folga
  A        2    0    2    0    2      0
  B        3    0    3    1    4      1
  C        2    2    4    2    4      0
  D        4    3    7    4    8      1
  E        4    4    8    4    8      0
  F        3    4    7   10   13      6
  G        5

## 2.2 Planeamento de projetos: PERT

Exemplo [`ex02_2_pert.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/02_project_planning/02_2_pert/ex02_2_pert.py) · funções da UC: [`pm.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/02_project_planning/02_1_cpm/pm.py), [`pert.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/02_project_planning/02_2_pert/pert.py)

```text
Reproduz os exemplos do deck 2.2 (planeamento de projetos: PERT).

Slide «Quando as durações são incertas»: B com (2, 3, 10) => te = 4, var = 1.78.
Laboratório com três estimativas: T = 13, crítico A, C, D, F, H, var = 0.556, sd = 0.745;
P(T <= 14) = P(Z <= 1.34) = 0.91; A--B--F--H tem var 2.111 e P(A--B--F--H <= 14) = 0.92.
Slide «Simulação: 200 000 projetos» (Beta-PERT, semente 2026): média 13.2;
P(T <= 13, 14, 15) = 41%, 80%, 96% (PERT: 50%, 91%, 99.6%); A--B--F--H é o mais longo em 23%.
Para resolver na aula: Exemplo PERT, T = 17, crítico C, F, H, J, P(T <= 22) = 0.989.

A simulação dá números diferentes noutro gerador (MATLAB/Octave): compara-se com tolerância.
```

In [3]:
import numpy as np

from pm import caminhos
from pert import Phi, pert, simula_pert


def mostra(titulo, P, alvos, xcam):
    """Tabela de te e var, caminho crítico, P(T <= x) para x em alvos e, por caminho, P(<= xcam)."""
    te, var, T, crit = pert(P)
    s2 = sum(var[a] for a in crit)
    print("\n" + titulo)
    print("  ativ.  to  tm  tp     te    var")
    for a, v in P.items():
        print("  %-5s %3g %3g %3g %6.4g %6.3f" % (a, v[1], v[2], v[3], te[a], var[a]))
    print("  duração esperada: %g semanas; caminho crítico: %s" % (T, ", ".join(crit)))
    print("  var = %.3f; sd = %.3f" % (s2, s2 ** 0.5))
    prob = []
    for x in alvos:
        z = (x - T) / s2 ** 0.5
        prob.append(Phi(z))
        print("  P(T <= %g) = Phi(%.3f) = %.4f" % (x, z, Phi(z)))
    cam = []
    for p in caminhos({a: (v[0], te[a]) for a, v in P.items()}):
        mu, v = sum(te[x] for x in p), sum(var[x] for x in p)
        cam.append(("--".join(p), mu, v))
        print("  caminho %-14s mu = %-3g var = %.3f  P(<= %g) = %.4f"
              % ("--".join(p), mu, v, xcam, Phi((xcam - mu) / v ** 0.5)))
    return te, var, T, crit, s2, prob, cam


ok = True
print("Complementos de IO — deck 2.2: planeamento de projetos (PERT)")

# ------------------------------------------------------------ uma atividade
teB, vB = (2 + 4 * 3 + 10) / 6, ((10 - 2) / 6) ** 2
print("\nEncomenda do equipamento (B): to = 2, tm = 3, tp = 10 -> te = %g, var = %.3f" % (teB, vB))
ok &= teB == 4 and abs(vB - 1.778) < 1e-3

# ------------------------------------------------------------ o laboratório
LAB = {"A": ([], 2, 3, 4), "B": (["A"], 2, 3, 10), "C": (["A"], 1, 2, 3), "D": (["C"], 2, 3, 4),
       "E": (["C"], 1, 2, 3), "F": (["B", "D", "E"], 2, 3, 4), "G": (["D"], 1, 2, 3), "H": (["F", "G"], 1, 2, 3)}
te, var, T, crit, s2, prob, cam = mostra("Laboratório com três estimativas", LAB, [13, 14, 15], 14)
ok &= (np.allclose([te[a] for a in LAB], [3, 4, 2, 3, 2, 3, 2, 2]) and abs(T - 13) < 1e-9
       and crit == ["A", "C", "D", "F", "H"] and abs(s2 - 0.556) < 1e-3 and abs(s2 ** 0.5 - 0.745) < 1e-3
       and round(1 / s2 ** 0.5, 2) == 1.34 and round(prob[1], 2) == 0.91
       and round(prob[0], 3) == 0.5 and round(prob[2], 3) == 0.996)
abfh = [c for c in cam if c[0] == "A--B--F--H"][0]
ok &= abs(abfh[1] - 12) < 1e-9 and abs(abfh[2] - 2.111) < 1e-3 and round(Phi(2 / abfh[2] ** 0.5), 2) == 0.92

# ------------------------------------------------------------ simulação
N = 200_000
Tsim, D = simula_pert(LAB, N, 2026)
psim = [(Tsim <= x).mean() for x in (13, 14, 15)]
print("\nSimulação: %d projetos, durações Beta-PERT, semente 2026" % N)
print("                 PERT    simulação")
print("  duração média  %.1f    %.1f" % (T, Tsim.mean()))
for x, pp, ps in zip((13, 14, 15), prob, psim):
    print("  P(T <= %g)     %.1f%%   %.0f%%" % (x, 100 * pp, 100 * ps))
paths = caminhos(LAB)
L = np.vstack([sum(D[x] for x in p) for p in paths])
freq = [(L.argmax(0) == i).mean() for i in range(len(paths))]
print("  caminho mais longo nos sorteios: "
      + "; ".join("%s %.0f%%" % ("--".join(p), 100 * f) for p, f in zip(paths, freq)))
# histograma do slide (densidade por intervalo de meia semana, de 10 a 18)
hist_slide = [0.001, 0.008, 0.0404, 0.1248, 0.2606, 0.3938, 0.427, 0.3468,
              0.2122, 0.1014, 0.046, 0.0212, 0.0098, 0.0044, 0.0018, 0.0004]
dens = [((Tsim >= x) & (Tsim < x + 0.5)).mean() / 0.5 for x in np.arange(10, 18, 0.5)]
hist_ok = max(abs(h - s) for h, s in zip(dens, hist_slide)) < 0.01
print("  histograma igual ao do slide (tolerância 0.01): %s" % ("sim" if hist_ok else "não"))
ok &= (abs(Tsim.mean() - 13.2) < 0.05 and all(abs(p - s) < 0.01 for p, s in zip(psim, (0.41, 0.80, 0.96)))
       and abs(freq[0] - 0.23) < 0.01 and hist_ok)

# ------------------------------------------------------------ para resolver na aula
PEX = {"A": ([], 5, 6, 7), "B": ([], 1, 3, 5), "C": ([], 1, 4, 7), "D": (["A"], 1, 2, 3),
       "E": (["B"], 1, 2, 9), "F": (["C"], 1, 5, 9), "G": (["C"], 2, 2, 8), "H": (["E", "F"], 4, 4, 10),
       "I": (["D"], 2, 5, 8), "J": (["H", "G"], 2, 2, 8)}
te2, var2, T2, crit2, s22, prob2, cam2 = mostra("Exemplo PERT (para resolver na aula)", PEX, [22], 22)
ok &= (abs(T2 - 17) < 1e-9 and crit2 == ["C", "F", "H", "J"] and abs(s22 - 4.778) < 1e-3
       and round(prob2[0], 3) == 0.989
       and [(c[0], round(c[1], 6), round(c[2], 3)) for c in cam2]
       == [("A--D--I", 13, 1.222), ("B--E--H--J", 14, 4.222), ("C--F--H--J", 17, 4.778), ("C--G--J", 10, 3)])

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Complementos de IO — deck 2.2: planeamento de projetos (PERT)

Encomenda do equipamento (B): to = 2, tm = 3, tp = 10 -> te = 4, var = 1.778

Laboratório com três estimativas
  ativ.  to  tm  tp     te    var
  A       2   3   4      3  0.111
  B       2   3  10      4  1.778
  C       1   2   3      2  0.111
  D       2   3   4      3  0.111
  E       1   2   3      2  0.111
  F       2   3   4      3  0.111
  G       1   2   3      2  0.111
  H       1   2   3      2  0.111
  duração esperada: 13 semanas; caminho crítico: A, C, D, F, H
  var = 0.556; sd = 0.745
  P(T <= 13) = Phi(0.000) = 0.5000
  P(T <= 14) = Phi(1.342) = 0.9101
  P(T <= 15) = Phi(2.683) = 0.9964
  caminho A--B--F--H     mu = 12  var = 2.111  P(<= 14) = 0.9157
  caminho A--C--D--F--H  mu = 13  var = 0.556  P(<= 14) = 0.9101
  caminho A--C--D--G--H  mu = 12  var = 0.556  P(<= 14) = 0.9964
  caminho A--C--E--F--H  mu = 12  var = 0.556  P(<= 14) = 0.9964



Simulação: 200000 projetos, durações Beta-PERT, semente 2026
                 PERT    simulação
  duração média  13.0    13.2
  P(T <= 13)     50.0%   41%
  P(T <= 14)     91.0%   80%
  P(T <= 15)     99.6%   96%


  caminho mais longo nos sorteios: A--B--F--H 23%; A--C--D--F--H 73%; A--C--D--G--H 2%; A--C--E--F--H 2%
  histograma igual ao do slide (tolerância 0.01): sim

Exemplo PERT (para resolver na aula)
  ativ.  to  tm  tp     te    var
  A       5   6   7      6  0.111
  B       1   3   5      3  0.444
  C       1   4   7      4  1.000
  D       1   2   3      2  0.111
  E       1   2   9      3  1.778
  F       1   5   9      5  1.778
  G       2   2   8      3  1.000
  H       4   4  10      5  1.000
  I       2   5   8      5  1.000
  J       2   2   8      3  1.000
  duração esperada: 17 semanas; caminho crítico: C, F, H, J
  var = 4.778; sd = 2.186
  P(T <= 22) = Phi(2.287) = 0.9889
  caminho A--D--I        mu = 13  var = 1.222  P(<= 22) = 1.0000
  caminho B--E--H--J     mu = 14  var = 4.222  P(<= 22) = 1.0000
  caminho C--F--H--J     mu = 17  var = 4.778  P(<= 22) = 0.9889
  caminho C--G--J        mu = 10  var = 3.000  P(<= 22) = 1.0000

confere com os slides: sim


## 2.3 Diagrama de Gantt e aceleração de projetos (crashing)

Exemplo [`ex02_3_gantt_crashing.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/02_project_planning/02_3_gantt_crashing/ex02_3_gantt_crashing.py) · funções da UC: [`pm.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/02_project_planning/02_1_cpm/pm.py), [`crashing.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/02_project_planning/02_3_gantt_crashing/crashing.py)

```text
Reproduz os exemplos do deck 2.3 (diagrama de Gantt e crashing).

Slide «Do plano ao calendário»: Gantt do laboratório (críticas A, C, D, F, H; folga em B, E e G).
Slide «E se o cliente quiser o laboratório mais cedo?»: custo por semana 3, 3, 2, 4, -, 4, 3, 5;
custo normal 63; bónus de 6 por semana antecipada.
Slides «Aceleração semana a semana» e «Custo total»: encurtar C, A, F, H (2, 3, 4, 5);
custo total 63, 59, 56, 54, 53, 54 para T = 13, ..., 8; ótimo T = 9 (53), porque 9 -> 8 custa 7 > 6.
Slide «Complemento»: a programação linear dá o mesmo (T = 9, custo 53).
Para resolver na aula: exame de 26/07/2019, T = 18 dias, crítico A, C, E, custo 3700;
crashing para 16 dias: 4050 (A, B e C menos 1 dia); Gantt do plano final.
```

In [4]:
from pm import cpm, caminhos
from crashing import gantt, crash, crash_pl


def acelera(titulo, acts, bonus, Tmin):
    """Tabela do crashing, da duração normal até Tmin: o que se encurta, custos e caminhos."""
    slope, b = crash(acts, bonus)
    Tn = max(b)
    cam = caminhos(acts)
    print("\n" + titulo)
    print("  custo por unidade: " + "; ".join("%s %s" % (a, ("%g" % s) if acts[a][1] > acts[a][3] else "-")
                                            for a, s in slope.items()))
    print("  caminhos: " + "  ".join("(%d) %s" % (k + 1, "--".join(p)) for k, p in enumerate(cam)))
    print("   T  encurtada   +custo  direto   bónus   total   duração dos caminhos")
    ant = None
    for T in sorted((T for T in b if T >= Tmin), reverse=True):
        c, d, tot = b[T]
        if ant is None:
            enc, marg = "---", "---"
        else:
            menos = [a for a in acts if d[a] < ant[1][a]]
            mais = [a for a in acts if d[a] > ant[1][a]]
            enc = "+".join(menos) + "".join(" repõe " + a for a in mais)
            marg = "%g" % (c - ant[0])
        print("  %2g  %-10s %6s %7g %7g %7g   %s" % (T, enc, marg, c, -bonus * (Tn - T), tot,
                                                 " ".join("%g" % sum(d[x] for x in p) for p in cam)))
        ant = (c, d)
    return slope, b


ok = True
print("Complementos de IO — deck 2.3: diagrama de Gantt e aceleração de projetos (crashing)")

# ------------------------------------------------------------ o laboratório
LAB = {"A": ([], 3, 6, 2, 9), "B": (["A"], 4, 20, 2, 26), "C": (["A"], 2, 4, 1, 6), "D": (["C"], 3, 8, 2, 12),
       "E": (["C"], 2, 5, 2, 5), "F": (["B", "D", "E"], 3, 10, 1, 18), "G": (["D"], 2, 6, 1, 9),
       "H": (["F", "G"], 2, 4, 1, 9)}
print("\nGantt do laboratório (# crítica, = com folga, - folga)")
g = gantt(LAB)
print("\n".join(g))
ok &= g[1:] == ["  A       ###..........", "  B       ...====-.....", "  C       ...##........",
                "  D       .....###.....", "  E       .....==-.....", "  F       ........###..",
                "  G       ........==-..", "  H       ...........##"]

slope, b = acelera("Crashing do laboratório (bónus 6 por semana antecipada)", LAB, 6, 8)
custo_normal = sum(v[2] for v in LAB.values())
tot = {T: v[2] for T, v in b.items()}
Topt = min(tot, key=lambda T: (tot[T], -T))
print("  custo normal %g; ótimo: T = %g semanas, custo total %g; %g -> %g custaria %g > %g"
      % (custo_normal, Topt, tot[Topt], Topt, Topt - 1, b[Topt - 1][0] - b[Topt][0], 6))
ok &= ([slope[a] for a in LAB] == [3, 3, 2, 4, 0, 4, 3, 5] and custo_normal == 63
       and [b[T][0] for T in range(13, 7, -1)] == [63, 65, 68, 72, 77, 84]
       and [tot[T] for T in range(13, 7, -1)] == [63, 59, 56, 54, 53, 54] and Topt == 9
       and [LAB[a][1] - b[9][1][a] for a in LAB] == [1, 0, 1, 0, 0, 1, 0, 1])

try:
    Tpl, dir_pl, tot_pl, y = crash_pl(LAB, 6)
    print("  programação linear (linprog do scipy): T = %.4g, custo direto %.4g, total %.4g"
          % (Tpl, dir_pl, tot_pl))
    ok &= abs(Tpl - 9) < 1e-6 and abs(tot_pl - 53) < 1e-6
except ImportError:
    print("  programação linear: precisa de scipy (pip install scipy)")

# ------------------------------------------------------------ para resolver na aula
EX = {"A": ([], 4, 700, 2, 1100), "B": ([], 9, 900, 7, 1000), "C": (["A"], 9, 300, 8, 400),
      "D": (["A", "B"], 3, 400, 2, 475), "E": (["C", "D"], 5, 600, 5, 600), "F": (["B", "D"], 4, 800, 3, 950)}
ES, EF, LS, LF, F, T = cpm(EX)
crit = [a for a in EX if abs(F[a]) < 1e-9]
print("\nExercício (exame de 26/07/2019), durações em dias")
print("  folgas: " + "; ".join("%s %g" % (a, F[a]) for a in EX))
print("  duração do projeto: %g dias; caminho crítico: %s; custo normal %g"
      % (T, ", ".join(crit), sum(v[2] for v in EX.values())))
slope2, b2 = acelera("Crashing do exercício até 16 dias", EX, 0, 16)
enc16 = [a for a in EX if b2[16][1][a] < EX[a][1]]
print("  16 dias: custo %g (encurtar %s um dia cada)" % (b2[16][0], ", ".join(enc16)))
print("Gantt do plano final (16 dias)")
g2 = gantt(EX, b2[16][1])
print("\n".join(g2))
ok &= (T == 18 and crit == ["A", "C", "E"] and [F[a] for a in EX] == [0, 1, 0, 1, 0, 2]
       and b2[18][0] == 3700 and b2[17][0] == 3800 and b2[16][0] == 4050 and enc16 == ["A", "B", "C"]
       and [slope2[a] for a in EX] == [200, 50, 100, 75, 0, 150]
       and g2[1:] == ["  A       ###.............", "  B       ########........", "  C       ...########.....",
                      "  D       ........###.....", "  E       ...........#####", "  F       ...........====-"])

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Complementos de IO — deck 2.3: diagrama de Gantt e aceleração de projetos (crashing)

Gantt do laboratório (# crítica, = com folga, - folga)
  tempo   1234567890123
  A       ###..........
  B       ...====-.....
  C       ...##........
  D       .....###.....
  E       .....==-.....
  F       ........###..
  G       ........==-..
  H       ...........##

Crashing do laboratório (bónus 6 por semana antecipada)
  custo por unidade: A 3; B 3; C 2; D 4; E -; F 4; G 3; H 5
  caminhos: (1) A--B--F--H  (2) A--C--D--F--H  (3) A--C--D--G--H  (4) A--C--E--F--H
   T  encurtada   +custo  direto   bónus   total   duração dos caminhos
  13  ---           ---      63       0      63   12 13 12 12
  12  C               2      65      -6      59   12 12 11 11
  11  A               3      68     -12      56   11 11 10 10
  10  F               4      72     -18      54   10 10 10 9
   9  H               5      77     -24      53   9 9 9 8
   8  B+D             7      84     -30      54   8 8 8 8
  custo

  programação linear (linprog do scipy): T = 9, custo direto 77, total 53

Exercício (exame de 26/07/2019), durações em dias
  folgas: A 0; B 1; C 0; D 1; E 0; F 2
  duração do projeto: 18 dias; caminho crítico: A, C, E; custo normal 3700

Crashing do exercício até 16 dias
  custo por unidade: A 200; B 50; C 100; D 75; E -; F 150
  caminhos: (1) A--C--E  (2) A--D--E  (3) A--D--F  (4) B--D--E  (5) B--D--F  (6) B--F
   T  encurtada   +custo  direto   bónus   total   duração dos caminhos
  18  ---           ---    3700       0    3700   18 12 11 17 16 13
  17  C             100    3800       0    3800   17 12 11 17 16 13
  16  A+B           250    4050       0    4050   16 11 10 16 15 12
  16 dias: custo 4050 (encurtar A, B, C um dia cada)
Gantt do plano final (16 dias)
  tempo   1234567890123456
  A       ###.............
  B       ########........
  C       ...########.....
  D       ........###.....
  E       ...........#####
  F       ...........====-

confere com os slides: sim
